# 03 — Cross-Reference: SQLite ↔ ChromaDB

The `chunks` table stores a `chroma_id` for every PDF chunk so that a SQLite
row can always be traced back to its ChromaDB vector.

This notebook demonstrates that linkage and shows how to look up a chunk
in ChromaDB using the ID stored in SQLite.

In [1]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

# Make project root importable so we can reuse SIRA's config
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DB_PATH = PROJECT_ROOT / 'data' / 'stocks.db'
assert DB_PATH.exists(), f'Database not found at {DB_PATH}'
con = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)

Connected to /home/guimas/Documents/Software/sira/data/stocks.db


## Full registry JOIN

Every chunk row joined to its parent document.

In [2]:
full = pd.read_sql(
    """
    SELECT
        d.filename,
        d.type,
        d.ingested_at,
        c.id          AS chunk_row_id,
        c.chroma_id,
        c.page,
        c.chunk_index
    FROM documents d
    JOIN chunks c ON c.document_id = d.id
    ORDER BY d.filename, c.page, c.chunk_index
    """,
    con,
)
print(f'{len(full)} chunk rows total')
full.head(20)

0 chunk rows total


,filename,type,ingested_at,chunk_row_id,chroma_id,page,chunk_index


## All chunk IDs for a given document

Change `TARGET_FILE` to any PDF filename shown above.

In [3]:
TARGET_FILE = full['filename'].iloc[0] if not full.empty else None

if TARGET_FILE:
    ids_for_doc = pd.read_sql(
        """
        SELECT c.chroma_id, c.page, c.chunk_index
        FROM chunks c
        JOIN documents d ON d.id = c.document_id
        WHERE d.filename = ?
        ORDER BY c.page, c.chunk_index
        """,
        con,
        params=(TARGET_FILE,),
    )
    print(f'{len(ids_for_doc)} chunks for "{TARGET_FILE}"')
    ids_for_doc.head(10)
else:
    print('No documents in registry yet.')

No documents in registry yet.


## Look up a chunk in ChromaDB

Using a `chroma_id` from the SQLite registry, fetch the actual text and
metadata from ChromaDB.

In [4]:
if TARGET_FILE and not ids_for_doc.empty:
    import chromadb
    from config import settings

    chroma_client = chromadb.PersistentClient(path=settings.chroma_persist_dir)
    collection = chroma_client.get_or_create_collection('pdf_chunks')

    sample_id = ids_for_doc['chroma_id'].iloc[0]
    result = collection.get(ids=[sample_id], include=['documents', 'metadatas'])

    print(f'chroma_id : {sample_id}')
    print(f'metadata  : {result["metadatas"][0]}')
    print(f'text      :\n{result["documents"][0][:300]}...')
else:
    print('No chunk IDs available — ingest a PDF first.')

No chunk IDs available — ingest a PDF first.


## Summary: how the linkage works

| SQLite column | Meaning |
|---|---|
| `chunks.chroma_id` | Primary key used in `collection.get(ids=[...])` |
| `chunks.document_id` | FK to `documents.id` — parent file |
| `chunks.page` | PDF page the chunk came from |
| `chunks.chunk_index` | Zero-based position within the page |

To delete all vectors for a document, SIRA calls
`collection.delete(where={"source": filename})` — which matches
the `metadata.source` field stored when the chunk was upserted.

In [5]:
con.close()